In [1]:
%load_ext autoreload
%autoreload 2

# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [1]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [2]:
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)

In [3]:
orders

,order_id,wait_time,expected_wait_time,delay_vs_expected,order_status,dim_is_five_star,dim_is_one_star,review_score,number_of_items,number_of_sellers,price,freight_value,distance_seller_customer
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,15.544063,0.0,delivered,0,0,4,1,1,29.99,8.72,18.063837
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,19.137766,0.0,delivered,0,0,4,1,1,118.70,22.76,856.292580
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,26.639711,0.0,delivered,1,0,5,1,1,159.90,19.22,514.130333
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,26.188819,0.0,delivered,1,0,5,1,1,45.00,27.20,1822.800366
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,12.112049,0.0,delivered,1,0,5,1,1,19.90,8.72,30.174037
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95875,9c5dedf39a927c1b2549525ed64a053c,8.218009,18.587442,0.0,delivered,1,0,5,1,1,72.00,13.08,69.481037
95876,63943bddc261676b46f01ca7ac2f7bd8,22.193727,23.459051,0.0,delivered,0,0,4,1,1,174.90,20.10,474.098245
95877,83c1379a015df1e13d02aae0204711ab,24.859421,30.384225,0.0,delivered,1,0,5,1,1,205.99,65.02,968.051192
95878,11c177c8e97725db2631073c19f07b62,17.086424,37.105243,0.0,delivered,0,0,2,2,1,359.98,81.18,370.146853


👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

In [5]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [9]:
orders_num=orders.select_dtypes(include="number", exclude=None)


🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [10]:
vif_data = pd.DataFrame()
vif_data["feature"] = orders_num.columns

vif_data["VIF"] = [variance_inflation_factor(orders_num.values, i)
                          for i in range(len(orders_num.columns))]
print(vif_data)

                     feature        VIF
0                  wait_time   8.676473
1         expected_wait_time  13.232158
2          delay_vs_expected   2.524797
3           dim_is_five_star   8.037115
4            dim_is_one_star   3.411968
5               review_score  60.417539
6            number_of_items   7.664078
7          number_of_sellers  45.504190
8                      price   1.732193
9              freight_value   3.549336
10  distance_seller_customer   3.230427


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

from sklearn.preprocessing import StandardScaler 

In [11]:
from sklearn.preprocessing import StandardScaler

In [14]:
scaler = StandardScaler().set_output(transform="pandas")
orders_num_scaled = scaler.fit_transform(orders_num)

vif_data_scaled = pd.DataFrame()
vif_data_scaled["feature"] = orders_num_scaled.columns

vif_data_scaled["VIF"] = [variance_inflation_factor(orders_num_scaled.values, i)
                          for i in range(len(orders_num_scaled.columns))]
print(vif_data_scaled)
to_drop=vif_data_scaled.loc[vif_data_scaled["VIF"]>10,"feature"]
print(to_drop)
# Remove highly correlated features
#corr_matrix = pd.DataFrame(X_train).corr().abs()
#upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
#to_drop = [column for column in orders_num_scaled.columns if any(upper[column] > 0.9)]

                     feature        VIF
0                  wait_time   3.229729
1         expected_wait_time   1.607475
2          delay_vs_expected   2.466969
3           dim_is_five_star   4.942408
4            dim_is_one_star   5.315187
5               review_score  12.130217
6            number_of_items   1.384804
7          number_of_sellers   1.109713
8                      price   1.209265
9              freight_value   1.679645
10  distance_seller_customer   1.603153
5    review_score
Name: feature, dtype: object


## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [19]:
 orders_num_scaled.dim_is_one_star

0       -0.328966
1       -0.328966
2       -0.328966
3       -0.328966
4       -0.328966
           ...   
95875   -0.328966
95876   -0.328966
95877   -0.328966
95878   -0.328966
95879   -0.328966
Name: dim_is_one_star, Length: 95872, dtype: float64

In [22]:
 logit_one= sm.Logit( orders.dim_is_one_star,orders.review_score ).fit()

Optimization terminated successfully.
         Current function value: 0.142986
         Iterations 8


`Logit 5️⃣`

In [23]:
 logit_five= sm.Logit( orders.dim_is_five_star,orders.review_score ).fit()

Optimization terminated successfully.
         Current function value: 0.607829
         Iterations 5


💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [24]:
print(logit_one.summary())

                           Logit Regression Results                           
Dep. Variable:        dim_is_one_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95871
Method:                           MLE   Df Model:                            0
Date:                Mon, 23 Feb 2026   Pseudo R-squ.:                  0.5530
Time:                        12:48:00   Log-Likelihood:                -13708.
converged:                       True   LL-Null:                       -30669.
Covariance Type:            nonrobust   LLR p-value:                       nan
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
review_score    -0.9273      0.006   -154.420      0.000      -0.939      -0.916


In [25]:
print(logit_five.summary())

                           Logit Regression Results                           
Dep. Variable:       dim_is_five_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95871
Method:                           MLE   Df Model:                            0
Date:                Mon, 23 Feb 2026   Pseudo R-squ.:                  0.1010
Time:                        12:48:43   Log-Likelihood:                -58274.
converged:                       True   LL-Null:                       -64817.
Covariance Type:            nonrobust   LLR p-value:                       nan
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
review_score     0.2005      0.002    121.102      0.000       0.197       0.204


In [27]:
logit_five.params

review_score    0.200519
dtype: float64

In [31]:
logit_one_2= sm.Logit( orders.dim_is_one_star,orders[["delay_vs_expected","wait_time"]] ).fit()
logit_five_2= sm.Logit( orders.dim_is_five_star,orders[["delay_vs_expected","wait_time"]] ).fit()

Optimization terminated successfully.
         Current function value: 0.404236
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.671259
         Iterations 7


In [33]:
print(logit_one_2.summary())
print(logit_five_2.summary())

                           Logit Regression Results                           
Dep. Variable:        dim_is_one_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95870
Method:                           MLE   Df Model:                            1
Date:                Mon, 23 Feb 2026   Pseudo R-squ.:                 -0.2637
Time:                        13:25:22   Log-Likelihood:                -38755.
converged:                       True   LL-Null:                       -30669.
Covariance Type:            nonrobust   LLR p-value:                     1.000
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
delay_vs_expected     0.4754      0.004    107.608      0.000       0.467       0.484
wait_time            -0.1744      0.001   -169.491      0.000      -0.176      -0.172
                        

In [38]:
# Aşağıdaki cümlelerden doğru olanları aşağıdaki listeye kaydedin.

a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = [a]

🧪 __Kodunu Test Et__

In [39]:
from nbresult import ChallengeResult

result = ChallengeResult('logit',
    answers = your_answer
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/scorp08/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/scorp08/.workintech/sprint-15/Logistic-Regression/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!